In [1]:
"""
Custom Cost-Based Evaluation Metric for Mercor's Kaggle Cheating Detection Competition
"""

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculate cost-based score for cheating detection.
    
    Finds optimal decision thresholds that divide predictions into three regions:
    - Auto-pass (low confidence cheating): $0 if correct, $600 if missed cheating
    - Manual review (medium confidence cheating): $5 if cheating, $150 if wasted on legitimate user
    - Auto-block (high confidence cheating): $0 if correct, $300 if wrongly blocked legitimate user

    """
    merged = solution.merge(submission, on=row_id_column_name, how='inner')
    
    if merged.empty:
        raise ValueError("No matching IDs between solution and submission")
    
    if 'prediction' not in merged.columns:
        raise ValueError("Submission must have 'prediction' column")
    
    target_candidates = [c for c in merged.columns 
                        if c not in [row_id_column_name, 'prediction', 'Usage']]
    
    if len(target_candidates) != 1:
        raise ValueError(f"Cannot identify target column. Found: {target_candidates}")
    
    target_col = target_candidates[0]
    
    y_true = merged[target_col].values
    y_pred = merged['prediction'].values
    
    if not is_numeric_dtype(merged['prediction']):
        raise ValueError("Predictions must be numeric")
    if not np.all((y_pred >= 0) & (y_pred <= 1)):
        raise ValueError("Predictions must be between 0 and 1")
    
    y_true = np.array(y_true, copy=False)
    sort_order = np.argsort(y_pred)
    y_true = y_true[sort_order]
    
    c1 = (y_true * 745 - 150).cumsum()
    
    c2 = (y_true * 155 - 150).cumsum()
    
    total_cost = c1.min() + c2.min() + (y_true == 0).sum() * 300
    
    return float(-total_cost)
